# Notebook 03 – Network Construction and Analysis

**DigitAfrica Workshop 2026 – Identifying Epistemic Enclaves and Understanding Polarisation**

## Learning objectives
- Build a social or interaction network from tabular data
- Compute and interpret basic network metrics (degree, clustering, path length)
- Visualise the network

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

%matplotlib inline

DATA_PROC = Path("../data/processed")

## 1. Build a synthetic interaction network

> **TODO:** Replace synthetic edge generation with edges derived from the actual dataset.

In [ ]:
rng = np.random.default_rng(1)
n_nodes = 100

# Assign nodes to two groups
groups = {i: ("A" if i < 50 else "B") for i in range(n_nodes)}

# Preferential within-group connections (homophily)
G = nx.Graph()
G.add_nodes_from(groups.keys())
nx.set_node_attributes(G, groups, "group")

for u in range(n_nodes):
    for v in range(u + 1, n_nodes):
        same_group = groups[u] == groups[v]
        prob = 0.08 if same_group else 0.01
        if rng.random() < prob:
            G.add_edge(u, v)

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

## 2. Basic network metrics

In [ ]:
degrees = dict(G.degree())
print(f"Average degree      : {np.mean(list(degrees.values())):.2f}")
print(f"Density             : {nx.density(G):.4f}")
print(f"Average clustering  : {nx.average_clustering(G):.4f}")

# Largest connected component
lcc = G.subgraph(max(nx.connected_components(G), key=len)).copy()
print(f"LCC size            : {lcc.number_of_nodes()} nodes")
print(f"Average path length : {nx.average_shortest_path_length(lcc):.2f}")

## 3. Degree distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(list(degrees.values()), bins=20, color="steelblue", edgecolor="white")
ax.set_xlabel("Degree")
ax.set_ylabel("Count")
ax.set_title("Degree distribution")
plt.tight_layout()
plt.show()

## 4. Network visualisation

In [ ]:
color_map = ["steelblue" if groups[n] == "A" else "tomato" for n in G.nodes()]
pos = nx.spring_layout(G, seed=42)

fig, ax = plt.subplots(figsize=(9, 7))
nx.draw_networkx(
    G, pos=pos, node_color=color_map,
    node_size=60, with_labels=False,
    edge_color="grey", alpha=0.7, ax=ax
)
ax.set_title("Interaction network (blue = Group A, red = Group B)")
ax.axis("off")
plt.tight_layout()
plt.show()